In [1]:
from pathlib import Path
import pandas as pd

from data import BasinDataLake

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs")

root_dir = save_dir / 'datalakes' / 'training'
store = BasinDataLake(root_dir)

zarr_path = '/scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr'

In [2]:
store.compute_and_store_stats(zarr_path, overwrite=True)

Dask client started. Dashboard at: http://127.0.0.1:8787/status
Processing source: gauge...


Basins in gauge: 100%|██████████| 643/643 [06:56<00:00,  1.54it/s]


Processing source: glow-s...


Basins in glow-s: 100%|██████████| 214/214 [00:28<00:00,  7.41it/s]


Processing source: era5...


Basins in era5: 100%|██████████| 637/637 [3:13:34<00:00, 18.23s/it]  


Processing source: swot-river...


Basins in swot-river: 100%|██████████| 192/192 [10:58<00:00,  3.43s/it]


In [9]:
basins = pd.read_csv("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph/basin_lists/proto_all.txt", header=None)
basins = basins[0].to_list()[0:10]
sources = ['era5']

In [63]:
from tqdm import tqdm
import zarr

def consolidate_zarr(zarr_root: Path | str):
    zarr_root = Path(zarr_root)
    sources = [p for p in zarr_root.iterdir() if p.is_dir()]

    for source_path in sources:
        basin_paths = [p for p in source_path.iterdir() if p.is_dir()]
        print(f"Consolidating source: {source_path.name}...")

        for basin_path in tqdm(basin_paths, desc=f"Basins in {source_path.name}"):
            zarr.consolidate_metadata(basin_path)

consolidate_zarr(zarr_path)

Consolidating source: gauge...


Basins in gauge:   0%|          | 0/643 [00:00<?, ?it/s]/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/zarr/api/asynchronous.py:233: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
Basins in gauge: 100%|██████████| 643/643 [00:05<00:00, 127.52it/s]


Consolidating source: glow-s...


Basins in glow-s: 100%|██████████| 214/214 [00:01<00:00, 130.56it/s]


Consolidating source: era5...


Basins in era5: 100%|██████████| 637/637 [00:17<00:00, 37.08it/s]


Consolidating source: swot-river...


Basins in swot-river: 100%|██████████| 192/192 [00:07<00:00, 26.70it/s]


In [ ]:
store.export_to_zarr(zarr_path, workers=6, start_date='1980-01-01', end_date="2024-12-31")

In [ ]:
from tqdm import tqdm
import xarray as xr

root_path = Path('/scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr')


print(f"🔍 Starting validation of Zarr store at: {root_path}\n")

corrupted_paths = []

# Get all subdirectories in the root path, assuming they are sources
# source_dirs = [d for d in root_path.iterdir() if d.is_dir()]
source_dirs = [root_path / 'era5']

# Set up the outer progress bar for sources
for source_path in tqdm(source_dirs, desc="Processing Sources", unit="source"):
    source_name = source_path.name
    
    basin_dirs = [d for d in source_path.iterdir() if d.is_dir()]
    
    # Set up the inner progress bar for basins within the current source
    for basin_path in basin_dirs:
        try:
            # The core operation: try to open the Zarr group.
            # The 'with' statement ensures the store is properly closed.
            with xr.open_zarr(basin_path, consolidated=False):
                pass # If this succeeds, the group is valid.

        except ValueError as e:
            # This is the specific error we are looking for!
            if "conflicting sizes for dimension" in str(e):
                tqdm.write(f"❌ CORRUPTED: {basin_path.relative_to(root_path)}")
                corrupted_paths.append(basin_path)
            else:
                # It's a different kind of ValueError
                tqdm.write(f"❗️ OTHER ERROR (ValueError): {basin_path.relative_to(root_path)} -> {e}")

        except Exception as e:
            # Catch any other error (e.g., not a valid Zarr store, permissions issue)
            tqdm.write(f"❗️ OTHER ERROR ({type(e).__name__}): {basin_path.relative_to(root_path)} -> {e}")

print("\n" + "="*50)
print("✅ Validation Complete")
print("="*50)

if corrupted_paths:
    print(f"\nFound {len(corrupted_paths)} corrupted Zarr groups that need regeneration:")
    for path in corrupted_paths:
        print(f"  - {path}")
else:
    print("\n🎉 No corrupted Zarr groups found!")



In [ ]:
import shutil
for p in corrupted_paths:
    print(f"Removing {p}")
    shutil.rmtree(p)

In [ ]:
start_date='1980-01-01'
end_date="2024-12-31"
year_step = 50

date_range = pd.date_range(start_date, end_date, freq=f"{year_step}YE", inclusive="both")

# Ensure the last date is exactly end_date
if date_range[-1] != pd.Timestamp(end_date):
    date_range = date_range.append(pd.DatetimeIndex([end_date]))

for start, end in zip(date_range[:-1], date_range[1:]):
    print(f"{start} - {end}")

In [ ]:
import xarray as xr

ds = xr.open_dataset(
    "/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr/era5/UKEA-520894",
    engine="zarr",
)

ds

In [ ]:
import numpy as np

np.isnan(ds['e_mean'].values).any()

In [ ]:
list(ds.data_vars)

In [ ]:
time_gaps = {
    column: ds[[column]].to_array().isnull().any().item()
    for column in list(ds.data_vars)
}
time_gaps

In [ ]:
ds

In [ ]:
ds.isnull().any().any().values()

In [ ]:
p = Path(zarr_path)
groups = [d.name for d in p.iterdir() if d.is_dir()]
print(f"Discovered groups: {groups}")
group_datasets = [xr.open_zarr(p / g) for g in groups]
ds = xr.merge(group_datasets)
ds


In [ ]:
group_datasets[1]

In [ ]:
import xarray as xr 

zarr_path = '/work/pi_cjgleason_umass_edu/ted/swot-ml-zarr'
ds_gauge = xr.open_zarr(zarr_path, group='era5')
ds_gauge

In [ ]:
# Example: Select a 15-day period 
start_slice = '1990-06-01'
end_slice = '1990-06-15'

# .sel() is the standard way to perform label-based selection
data_slice = ds_gauge.sel(
    date=slice(start_slice, end_slice),
)
data_slice

In [ ]:
data_slice.load()

In [ ]:
tmp.index.get_level_values('date').tz_localize(None)

In [ ]:
def read_file(fp):
    with open(fp, "r") as file:
        basin_list = file.readlines()
        basin_list = [basin.strip() for basin in basin_list]
    return basin_list

list_dir = save_dir.parent / 'multigraph' / 'basin_lists'
train_basins = read_file(list_dir / 'proto_train.txt')
test_basins = read_file(list_dir / 'proto_test.txt')

In [ ]:
static_df = store.read_static()


subbasins = (
    static_df.loc[static_df["basin"] == basin, "subbasin"]
    .astype(str)
    .unique()
    .tolist()
)
subbasins = sorted(subbasins)
subbasins

In [ ]:
df = tmp
value_cols = [c for c in df.columns if c not in ["basin", "year"]]
value_cols

In [ ]:
import xarray as xr
ds_xr = xr.Dataset.from_dataframe(df)
ds_xr

In [ ]:
ds_xr.reindex(subbasin=subbasins)

In [ ]:
import dask.dataframe as dd
dd.from_pyarrow_dataset(pyarrow_dataset)

In [ ]:
pa.fragments

In [ ]:
tmp = store.read_dynamic(train_basins[-1], start_date = '2024-01-01', end_date = '2024-02-01')
tmp

In [ ]:
for col in tmp.columns:
    print(f"{col}: {tmp[col].dtype}")

In [ ]:
for idx, grp in tmp.groupby('subbasin'):
    grp.droplevel(0,0)['gauge']['discharge'].plot()